In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

import os
import warnings
warnings.filterwarnings(action='ignore', category=UserWarning)

plt.rc('font', family='NanumBarunGothic') # matplotlib 한글 깨짐 방지

## **0. 데이터 로드**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/날씨빅콘/train_heat.csv')

# 첫번째 열 제거
data = data.drop(columns=["Unnamed: 0"])

# 컬럼명에서 train_heat. 제거
data.columns = data.columns.str.replace("train_heat.", "", regex=False)

# 날짜 데이터를 데이트타임으로 변경
data['tm'] = pd.to_datetime(data['tm'].astype(str), format="%Y%m%d%H")

data.head()

,tm,branch_id,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi,heat_demand
0,2021-01-01 01:00:00,A,-10.1,78.3,0.5,0.0,0.0,68.2,-99.0,-8.2,281
1,2021-01-01 02:00:00,A,-10.2,71.9,0.6,0.0,0.0,69.9,-99.0,-8.6,262
2,2021-01-01 03:00:00,A,-10.0,360.0,0.0,0.0,0.0,69.2,-99.0,-8.8,266
3,2021-01-01 04:00:00,A,-9.3,155.9,0.5,0.0,0.0,65.0,-99.0,-8.9,285
4,2021-01-01 05:00:00,A,-9.0,74.3,1.9,0.0,0.0,63.5,-99.0,-9.2,283


* 파생 변수 생성

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

data['date'] = pd.to_datetime(data['tm']).dt.date
data['year'] = data['tm'].dt.year
data['month'] = data['tm'].dt.month
data['day'] = data['tm'].dt.day
data['hour'] = data['tm'].dt.hour
data['quarter'] = data['tm'].dt.quarter # 분기별
data['day_of_week'] = data['tm'].dt.dayofweek # 요일
data['season'] = data['month'].apply(get_season) # 계절
data['is_weekend'] = data['day_of_week'].apply(lambda x: 'Weekend' if x >= 5 else 'Weekday') # 주말 여부
#data['weekofyear'] = data['tm'].dt.isocalendar().week # 주차

# 3시간 단위 시간대 컬럼 생성 (0~23시 → 0~7로 구간화)
data['hour_group'] = (data['tm'].dt.hour // 3).astype(int)
hour_labels = ['0-3', '3-6', '6-9', '9-12', '12-15', '15-18', '18-21', '21-24']

In [ ]:
data.head()

,tm,branch_id,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi,...,date,year,month,day,hour,quarter,day_of_week,season,is_weekend,hour_group
0,2021-01-01 01:00:00,A,-10.1,78.3,0.5,0.0,0.0,68.2,-99.0,-8.2,...,2021-01-01,2021,1,1,1,1,4,Winter,Weekday,0
1,2021-01-01 02:00:00,A,-10.2,71.9,0.6,0.0,0.0,69.9,-99.0,-8.6,...,2021-01-01,2021,1,1,2,1,4,Winter,Weekday,0
2,2021-01-01 03:00:00,A,-10.0,360.0,0.0,0.0,0.0,69.2,-99.0,-8.8,...,2021-01-01,2021,1,1,3,1,4,Winter,Weekday,1
3,2021-01-01 04:00:00,A,-9.3,155.9,0.5,0.0,0.0,65.0,-99.0,-8.9,...,2021-01-01,2021,1,1,4,1,4,Winter,Weekday,1
4,2021-01-01 05:00:00,A,-9.0,74.3,1.9,0.0,0.0,63.5,-99.0,-9.2,...,2021-01-01,2021,1,1,5,1,4,Winter,Weekday,1


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499301 entries, 0 to 499300
Data columns (total 21 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   tm           499301 non-null  datetime64[ns]
 1   branch_id    499301 non-null  object        
 2   ta           486304 non-null  float64       
 3   wd           478897 non-null  float64       
 4   ws           480486 non-null  float64       
 5   rn_day       480675 non-null  float64       
 6   rn_hr1       480147 non-null  float64       
 7   hm           459584 non-null  float64       
 8   si           488954 non-null  float64       
 9   ta_chi       499281 non-null  float64       
 10  heat_demand  499278 non-null  float64       
 11  date         499301 non-null  object        
 12  year         499301 non-null  int32         
 13  month        499301 non-null  int32         
 14  day          499301 non-null  int32         
 15  hour         499301 non-null  int3

## **1. 결측치 NA 처리**

* -99 값은 결측치로 간주
* 이때 풍향(wd)는 -9.9도 결측치로 간주

In [ ]:
data['ta'] = data['ta'].replace(-99, pd.NA)
data['wd'] = data['wd'].replace([-99, -9.9], pd.NA)
data['ws'] = data['ws'].replace(-99, pd.NA)
data['rn_day'] = data['rn_day'].replace(-99, pd.NA)
data['rn_hr1'] = data['rn_hr1'].replace(-99, pd.NA)
data['hm'] = data['hm'].replace(-99, pd.NA)
data['si'] = data['si'].replace(-99, pd.NA)
data['ta_chi'] = data['ta_chi'].replace(-99, pd.NA)
data['heat_demand'] = data['heat_demand'].replace(-99, pd.NA)

In [ ]:
# 수치형으로 변환
cols = ['ta', 'wd', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'heat_demand']

for col in cols:
    data[col] = pd.to_numeric(data[col], errors='raise')

## **2. si 변수 보간하기**

* si 변수의 결측치는 총 232,922개로 다른 변수에 비해 많았으며, 대부분이 밤 시간대에 일사량이 존재하지 않아 발생한 것으로 확인되었다.

* 따라서 각 지점별(branch), 날짜별로 일사량이 관측되지 않는 시간대를 정리하였고, 해당 시간대의 결측값은 자동으로 0으로 대체되도록
구현하였다.

In [ ]:
# 공통 규칙 정의

# A,B,C,F
common_night_rules_1 = [
    ((1, 1), (2, 6), 19, 7),
    ((2, 6), (3, 4), 20, 7),
    ((3, 4), (4, 7), 20, 6),
    ((4, 7), (4, 13), 21, 6),
    ((4, 13), (8, 28), 21, 5),
    ((8, 28), (9, 3), 21, 6),
    ((9, 3), (10, 12), 20, 6),
    ((10, 12), (11, 3), 19, 6),
    ((11, 3), (12, 31), 19, 7)
]

# D,E,G,H,I,J,K,S
common_night_rules_2 = [
    ((1, 1), (2, 5), 19, 7),
    ((2, 5), (3, 4), 20, 7),
    ((3, 4), (4, 7), 20, 6),
    ((4, 7), (4, 14), 21, 6),
    ((4, 14), (8, 28), 21, 5),
    ((8, 28), (9, 3), 21, 6),
    ((9, 3), (10, 13), 20, 6),
    ((10, 13), (11, 4), 19, 6),
    ((11, 4), (12, 31), 19, 7)
]

# L,M,N
common_night_rules_3 = [
    ((1, 1), (2, 10), 19, 7),
    ((2, 10), (2, 25), 20, 7),
    ((2, 25), (4, 9), 20, 6),
    ((4, 9), (4, 19), 20, 5),
    ((4, 19), (8, 26), 21, 5),
    ((8, 26), (9, 4), 20, 5),
    ((9, 4), (10, 7), 20, 6),
    ((10, 7), (11, 13), 19, 6),
    ((11, 13), (12, 31), 19, 7)
]

# O
common_night_rules_4 = [
    ((1, 1), (2, 6), 19, 7),
    ((2, 6), (3, 2), 20, 7),
    ((3, 2), (4, 11), 20, 6),
    ((4, 11), (4, 13), 21, 6),
    ((4, 13), (8,30), 21, 5),
    ((8, 30), (9, 1), 21, 6),
    ((9, 1), (10, 11), 20, 6),
    ((10, 11), (11, 7), 19, 6),
    ((11, 7), (12, 31), 19, 7)
]

# P
common_night_rules_5 = [
    ((1, 1), (2, 6), 19, 7),
    ((2, 6), (3, 3), 20, 7),
    ((3, 3), (4, 10), 20, 6),
    ((4, 10), (4, 13), 21, 6),
    ((4, 13), (8,30), 21, 5),
    ((8, 30), (9, 2), 21, 6),
    ((9, 2), (10, 12), 20, 6),
    ((10, 12), (11, 6), 19, 6),
    ((11, 6), (12, 31), 19, 7)
]

# Q
common_night_rules_6 = [
    ((1, 1), (2, 5), 19, 7),
    ((2, 5), (3, 5), 20, 7),
    ((3, 5), (4, 7), 20, 6),
    ((4, 7), (4, 14), 21, 6),
    ((4, 14), (8, 28), 21, 5),
    ((8, 28), (9, 3), 21, 6),
    ((9, 3), (10, 13), 20, 6),
    ((10, 13), (11, 2), 19, 6),
    ((11, 2), (12, 31), 19, 7)
]

# R
common_night_rules_7 = [
    ((1, 1), (2, 1), 19, 7),
    ((2, 1), (3, 2), 20, 7),
    ((3, 2), (4, 9), 20, 6),
    ((4, 9), (4, 15), 21, 6),
    ((4, 15), (8, 26), 21, 5),
    ((8, 26), (9, 2), 21, 6),
    ((9, 2), (10, 14), 20, 6),
    ((10, 14), (11, 8), 19, 6),
    ((11, 8), (12, 31), 19, 7)
]


# 브랜치별 야간 시간 규칙 딕셔너리
branch_night_rules = {
    'A': common_night_rules_1,
    'B': common_night_rules_1,
    'C': common_night_rules_1,
    'D': common_night_rules_2,
    'E': common_night_rules_2,
    'F': common_night_rules_1,
    'G': common_night_rules_2,
    'H': common_night_rules_2,
    'I': common_night_rules_2,
    'J': common_night_rules_2,
    'K': common_night_rules_2,
    'L': common_night_rules_3,
    'M': common_night_rules_3,
    'N': common_night_rules_3,
    'O': common_night_rules_4,
    'P': common_night_rules_5,
    'Q': common_night_rules_6,
    'R': common_night_rules_7,
    'S': common_night_rules_2
}

In [ ]:
# 모든 브랜치에 대해 야간 시간 적용
for branch, rules in branch_night_rules.items():
    is_branch = data['branch_id'] == branch

    for (start_m, start_d), (end_m, end_d), night_start, night_end in rules:
        in_period = (
            ((data['month'] > start_m) | ((data['month'] == start_m) & (data['day'] >= start_d))) &
            ((data['month'] < end_m) | ((data['month'] == end_m) & (data['day'] <= end_d)))
        )

        if night_start > night_end:  # 자정을 넘기는 경우
            in_night = (data['hour'] >= night_start) | (data['hour'] <= night_end)
        else:
            in_night = (data['hour'] >= night_start) & (data['hour'] < night_end)

        # 조건에 맞는 행에 대해 si 값 0으로 설정
        mask = is_branch & in_period & in_night
        data.loc[mask, 'si'] = 0

## **3. 브랜치별 결측치 보간하기**

- branch별로 지역이 달라 feature 변수인 기상 요소들이 서로 상이하기 때문에 결측치를 branch별로 보간을 진행하였다.
- 단기 결측이나 적은 결측은 선형 보간, 장기 결측은 모델 기반 보간이 유리하다.
- 따라서 보간 방법을 정하기 위해 선형 보간, SoftImpute 보간, MICE 보간 3가지 방법을 각 브랜치에 적용하여 어떤 방법이 가장 성능이 좋은지 MAE를 기준으로 측정하고, 최적의 모델을 적용하였다.

In [ ]:
pip install fancyimpute

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.3 MB/s eta 0:00:00
  Created wheel for fancyimpute: filename=fancyimpute-0.7.0-py3-none-any.whl size=29879 sha256=0cdd59c078b004099bcc0ac9b12237121722ebc1894fd4926b92d57c685ee0cb
  Stored in directory: /root/.cache/pip/wheels/1a/f3/a1/f7f10b5ae2c2459398762a3fcf4ac18c325311c7e3163d5a15
  Created wheel for knnimpute: filename=knnimpute-0.1.0-py3-none-any.whl size=11331 sha256=c5b29c1d4ee5cb455a07786b975a8eea5685679a7a39321434644df82078bd43
  Stored in directory: /root/.cache/pip/wheels/ea/e8/e0/79872972161e54486517ae507f94b2c7cea27fb7ef793bd415
Successfully built fancyimpute knnimpute


In [ ]:
# branch_id 기준으로 나눔
branch_groups = {branch: sub_df for branch, sub_df in data.groupby('branch_id')}

df_A = branch_groups['A']
df_B = branch_groups['B']
df_C = branch_groups['C']
df_D = branch_groups['D']
df_E = branch_groups['E']
df_F = branch_groups['F']
df_G = branch_groups['G']
df_H = branch_groups['H']
df_I = branch_groups['I']
df_J = branch_groups['J']
df_K = branch_groups['K']
df_L = branch_groups['L']
df_M = branch_groups['M']
df_N = branch_groups['N']
df_O = branch_groups['O']
df_P = branch_groups['P']
df_Q = branch_groups['Q']
df_R = branch_groups['R']
df_S = branch_groups['S']

In [ ]:
# 수치형 변수
feature_cols = ['ta', 'wd', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi']
df_A_num = df_A[feature_cols]
df_B_num = df_B[feature_cols]
df_C_num = df_C[feature_cols]
df_D_num = df_D[feature_cols]
df_E_num = df_E[feature_cols]
df_F_num = df_F[feature_cols]
df_G_num = df_G[feature_cols]
df_H_num = df_H[feature_cols]
df_I_num = df_I[feature_cols]
df_J_num = df_J[feature_cols]
df_K_num = df_K[feature_cols]
df_L_num = df_L[feature_cols]
df_M_num = df_M[feature_cols]
df_N_num = df_N[feature_cols]
df_O_num = df_O[feature_cols]
df_P_num = df_P[feature_cols]
df_Q_num = df_Q[feature_cols]
df_R_num = df_R[feature_cols]
df_S_num = df_S[feature_cols]

In [ ]:
from sklearn.metrics import mean_absolute_error
from fancyimpute import SoftImpute, IterativeImputer
from tqdm import tqdm

# 보간 대상 데이터프레임 목록
branches = [chr(i) for i in range(ord('A'), ord('T'))]
imputed_results = {}
best_methods = {}

def calculate_mae(imputed_df, true_df, mask):
    return np.mean(np.abs(imputed_df.values[mask.values] - true_df.values[mask.values]))

for branch in tqdm(branches):
    df_name = f'df_{branch}_num'
    df = globals()[df_name].copy()
    nan_mask = df.isna()
    df_truth = df.copy()

    # 선형 보간
    df_linear = df.interpolate(method='linear', limit_direction='both')

    # SoftImpute 보간
    softimputer = SoftImpute(verbose=False)
    df_soft = pd.DataFrame(softimputer.fit_transform(df.values), columns=df.columns)

    # MICE 보간
    mice_imputer = IterativeImputer(verbose=False)
    df_mice = pd.DataFrame(mice_imputer.fit_transform(df.values), columns=df.columns)

    # MAE 계산
    mae_linear = calculate_mae(df_linear, df_truth.fillna(0), nan_mask)
    mae_soft = calculate_mae(df_soft, df_truth.fillna(0), nan_mask)
    mae_mice = calculate_mae(df_mice, df_truth.fillna(0), nan_mask)

    # MAE 정리
    maes = {
        'linear': mae_linear,
        'softimpute': mae_soft,
        'mice': mae_mice
    }

    best_method = min(maes, key=maes.get)
    best_methods[branch] = best_method

    # 최적 방법 결과 저장
    if best_method == 'linear':
        imputed_results[branch] = df_linear
    elif best_method == 'softimpute':
        imputed_results[branch] = df_soft
    elif best_method == 'mice':
        imputed_results[branch] = df_mice

    print(f"Branch {branch}: Best method = {best_method.upper()} (MAE: {maes[best_method]:.4f})")

  0%|          | 0/19 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
  5%|▌         | 1/19 [00:07<02:18,  7.71s/it]

Branch A: Best method = SOFTIMPUTE (MAE: 2.6269)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
 11%|█         | 2/19 [00:13<01:50,  6.49s/it]

Branch B: Best method = SOFTIMPUTE (MAE: 23.3904)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
 16%|█▌        | 3/19 [00:24<02:15,  8.46s/it]

...

Branch Q: Best method = SOFTIMPUTE (MAE: 39.1325)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
 95%|█████████▍| 18/19 [01:09<00:02,  2.17s/it]

Branch R: Best method = SOFTIMPUTE (MAE: 2.8941)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
100%|██████████| 19/19 [01:11<00:00,  3.77s/it]

Branch S: Best method = SOFTIMPUTE (MAE: 7.7535)


<브랜치별 최적 모델 및 MAE>

* Branch A: Best method = SOFTIMPUTE (MAE: 2.6269)

* Branch B: Best method = SOFTIMPUTE (MAE: 23.3904)

* Branch C: Best method = SOFTIMPUTE (MAE: 16.8799)

* Branch D: Best method = SOFTIMPUTE (MAE: 14.0001)

* Branch E: Best method = SOFTIMPUTE (MAE: 14.0001)

* Branch F: Best method = SOFTIMPUTE (MAE: 7.3781)

* Branch G: Best method = SOFTIMPUTE (MAE: 20.5615)

* Branch H: Best method = SOFTIMPUTE (MAE: 20.5615)

* Branch I: Best method = SOFTIMPUTE (MAE: 39.2930)

* Branch J: Best method = SOFTIMPUTE (MAE: 4.1853)

* Branch K: Best method = SOFTIMPUTE (MAE: 20.6671)

* Branch L: Best method = SOFTIMPUTE (MAE: 6.7703)

* Branch M: Best method = SOFTIMPUTE (MAE: 5.0910)

* Branch N: Best method = LINEAR (MAE: 16.5058)

* Branch O: Best method = SOFTIMPUTE (MAE: 2.1560)

* Branch P: Best method = SOFTIMPUTE (MAE: 0.6141)

* Branch Q: Best method = SOFTIMPUTE (MAE: 39.1325)

* Branch R: Best method = SOFTIMPUTE (MAE: 2.8941)

* Branch S: Best method = SOFTIMPUTE (MAE: 7.7535)

In [ ]:
# 결과 접근
# imputed_results['A'] → df_A_num 보간된 결과

imputed_results['A']

,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi
0,-10.1,78.3,0.5,0.0,0.0,68.2,0.0,-8.2
1,-10.2,71.9,0.6,0.0,0.0,69.9,0.0,-8.6
2,-10.0,360.0,0.0,0.0,0.0,69.2,0.0,-8.8
3,-9.3,155.9,0.5,0.0,0.0,65.0,0.0,-8.9
4,-9.0,74.3,1.9,0.0,0.0,63.5,0.0,-9.2
...,...,...,...,...,...,...,...,...
26274,1.1,3.8,0.5,2.5,0.0,95.3,0.0,2.0
26275,0.8,325.6,0.6,2.5,0.0,96.0,0.0,1.5
26276,0.0,331.2,0.4,2.5,0.0,96.3,0.0,1.2
26277,0.0,25.8,0.4,2.5,0.0,96.7,0.0,0.8


In [ ]:
# 열을 덮어쓸 branches 정의
branches = [chr(i) for i in range(ord('A'), ord('T'))]  # 'A' to 'S'
final_results = {}

for branch in branches:
    original_df = globals()[f'df_{branch}']
    imputed_df = imputed_results[branch]

    # imputed_df의 열만 original_df에서 대체
    filled_df = original_df.copy()
    for col in imputed_df.columns:
        filled_df[col] = imputed_df[col].values  # 행 순서 그대로 대입

    final_results[branch] = filled_df
    globals()[f'df_{branch}_filled'] = filled_df  # 이름으로도 저장

    print(f"{branch}: 보간된 {len(imputed_df.columns)}개 열을 덮어쓴 df_{branch}_filled 생성 완료")

A: 보간된 8개 열을 덮어쓴 df_A_filled 생성 완료
B: 보간된 8개 열을 덮어쓴 df_B_filled 생성 완료
C: 보간된 8개 열을 덮어쓴 df_C_filled 생성 완료
D: 보간된 8개 열을 덮어쓴 df_D_filled 생성 완료
E: 보간된 8개 열을 덮어쓴 df_E_filled 생성 완료
F: 보간된 8개 열을 덮어쓴 df_F_filled 생성 완료
G: 보간된 8개 열을 덮어쓴 df_G_filled 생성 완료
H: 보간된 8개 열을 덮어쓴 df_H_filled 생성 완료
I: 보간된 8개 열을 덮어쓴 df_I_filled 생성 완료
J: 보간된 8개 열을 덮어쓴 df_J_filled 생성 완료
K: 보간된 8개 열을 덮어쓴 df_K_filled 생성 완료
L: 보간된 8개 열을 덮어쓴 df_L_filled 생성 완료
M: 보간된 8개 열을 덮어쓴 df_M_filled 생성 완료
N: 보간된 8개 열을 덮어쓴 df_N_filled 생성 완료
O: 보간된 8개 열을 덮어쓴 df_O_filled 생성 완료
P: 보간된 8개 열을 덮어쓴 df_P_filled 생성 완료
Q: 보간된 8개 열을 덮어쓴 df_Q_filled 생성 완료
R: 보간된 8개 열을 덮어쓴 df_R_filled 생성 완료
S: 보간된 8개 열을 덮어쓴 df_S_filled 생성 완료


In [ ]:
df_A_filled.isna().sum()

,0
tm,0
branch_id,0
ta,0
wd,0
ws,0
rn_day,0
rn_hr1,0
hm,0
si,0
ta_chi,0


In [ ]:
# 모든 filled DataFrame을 리스트에 담기
all_filled_dfs = [globals()[f'df_{branch}_filled'] for branch in [chr(i) for i in range(ord('A'), ord('T'))]]

# 하나의 DataFrame으로 병합
df_all_filled = pd.concat(all_filled_dfs, axis=0, ignore_index=True)

# CSV로 저장
df_all_filled.to_csv('/content/drive/MyDrive/Colab Notebooks/날씨빅콘/weather_data_imputed2.csv', index=False)

## **4. 보간 후 처리**

In [ ]:
df_filled = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/날씨빅콘/weather_data_imputed2.csv')
df_filled

,tm,branch_id,ta,wd,ws,rn_day,rn_hr1,hm,si,ta_chi,...,date,year,month,day,hour,quarter,day_of_week,season,is_weekend,hour_group
0,2021-01-01 01:00:00,A,-10.1,78.3,0.5,0.0,0.0,68.2,0.0,-8.2,...,2021-01-01,2021,1,1,1,1,4,Winter,Weekday,0
1,2021-01-01 02:00:00,A,-10.2,71.9,0.6,0.0,0.0,69.9,0.0,-8.6,...,2021-01-01,2021,1,1,2,1,4,Winter,Weekday,0
2,2021-01-01 03:00:00,A,-10.0,360.0,0.0,0.0,0.0,69.2,0.0,-8.8,...,2021-01-01,2021,1,1,3,1,4,Winter,Weekday,1
3,2021-01-01 04:00:00,A,-9.3,155.9,0.5,0.0,0.0,65.0,0.0,-8.9,...,2021-01-01,2021,1,1,4,1,4,Winter,Weekday,1
4,2021-01-01 05:00:00,A,-9.0,74.3,1.9,0.0,0.0,63.5,0.0,-9.2,...,2021-01-01,2021,1,1,5,1,4,Winter,Weekday,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499296,2023-12-31 19:00:00,S,3.2,233.5,0.4,2.5,0.0,91.5,0.0,2.8,...,2023-12-31,2023,12,31,19,4,6,Winter,Weekend,6
499297,2023-12-31 20:00:00,S,2.9,227.4,0.1,2.5,0.0,92.1,0.0,2.7,...,2023-12-31,2023,12,31,20,4,6,Winter,Weekend,6
499298,2023-12-31 21:00:00,S,2.1,360.0,0.0,2.5,0.0,93.3,0.0,1.4,...,2023-12-31,2023,12,31,21,4,6,Winter,Weekend,7
499299,2023-12-31 22:00:00,S,2.2,30.0,1.4,2.5,0.0,95.5,0.0,1.3,...,2023-12-31,2023,12,31,22,4,6,Winter,Weekend,7


In [ ]:
df_filled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499301 entries, 0 to 499300
Data columns (total 21 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   tm           499301 non-null  object 
 1   branch_id    499301 non-null  object 
 2   ta           499301 non-null  float64
 3   wd           499301 non-null  float64
 4   ws           499301 non-null  float64
 5   rn_day       499301 non-null  float64
 6   rn_hr1       499301 non-null  float64
 7   hm           499301 non-null  float64
 8   si           499301 non-null  float64
 9   ta_chi       499301 non-null  float64
 10  heat_demand  499278 non-null  float64
 11  date         499301 non-null  object 
 12  year         499301 non-null  int64  
 13  month        499301 non-null  int64  
 14  day          499301 non-null  int64  
 15  hour         499301 non-null  int64  
 16  quarter      499301 non-null  int64  
 17  day_of_week  499301 non-null  int64  
 18  season       499301 non-

* target 변수 결측치는 총 23개로, 전체 데이터에 비해 비율이 매우 작으므로 결측 행은 삭제해주었다.

In [ ]:
# 타겟 na인 칼럼 삭제
df_filled = df_filled.dropna(subset=['heat_demand']).reset_index(drop=True)
df_filled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499278 entries, 0 to 499277
Data columns (total 21 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   tm           499278 non-null  object 
 1   branch_id    499278 non-null  object 
 2   ta           499278 non-null  float64
 3   wd           499278 non-null  float64
 4   ws           499278 non-null  float64
 5   rn_day       499278 non-null  float64
 6   rn_hr1       499278 non-null  float64
 7   hm           499278 non-null  float64
 8   si           499278 non-null  float64
 9   ta_chi       499278 non-null  float64
 10  heat_demand  499278 non-null  float64
 11  date         499278 non-null  object 
 12  year         499278 non-null  int64  
 13  month        499278 non-null  int64  
 14  day          499278 non-null  int64  
 15  hour         499278 non-null  int64  
 16  quarter      499278 non-null  int64  
 17  day_of_week  499278 non-null  int64  
 18  season       499278 non-

* 음수가 나오면 안 되는 칼럼: 풍향 (wd), 일사량 (si), 풍속 (ws), 강수량 (rn_day, rn_hr1), 습도(hm)

* 기상 데이터의 물리적 제약조건을 적용하여 풍향은 0~360도 범위로 정규화하고, 일사량·풍속·강수량·습도는 클리핑을 통해 0 이상 값으로 제한하여 비현실적인 값을 제거

In [ ]:
# 클리핑

# 풍향 범위는 0~360도
df_filled['wd'] = df_filled['wd'] % 360

# 일사량, 풍속, 강수량 등은 0 이상만 허용
for col in ['si', 'ws', 'rn_day', 'rn_hr1', 'hm']:
    df_filled[col] = df_filled[col].clip(lower=0)

In [ ]:
# 풍향 sin, cos 변환
df_filled['wd_rad'] = np.deg2rad(df_filled['wd'])  # 도 → 라디안 변환
df_filled['wd_sin'] = np.sin(df_filled['wd_rad'])
df_filled['wd_cos'] = np.cos(df_filled['wd_rad'])